In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

# Load & prep
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
df = pd.read_csv('../data/raw/marketing_campaign.csv', sep='\t')

print("First 5 records:", df.head())
df = df.dropna(subset=['Income'])
df['Age'] = 2014 - df['Year_Birth']
df['Total_Spending'] = df['MntWines'] + df['MntFruits'] + df['MntMeatProducts'] + df['MntFishProducts'] + df['MntSweetProducts'] + df['MntGoldProds']
df['Total_Purchases'] = df['NumWebPurchases'] + df['NumCatalogPurchases'] + df['NumStorePurchases']

# Target: CLV proxy (total spending is our best proxy for customer value)
y = df['Total_Spending']

# Features: demographics + behavioral (NO spending columns — that's the target)
numeric_features = ['Income', 'Recency', 'Age', 'NumDealsPurchases','NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth', 'Kidhome', 'Teenhome']
categorical_features = ['Education', 'Marital_Status']

X = df[numeric_features + categorical_features]

# Preprocessor (from production/labs/03b_pipeline.ipynb pattern)
preprocessor = ColumnTransformer(transformers=[
  ('num', StandardScaler(), numeric_features),
  ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_features)
])

# Three models to compare
models = {
  'Linear Regression': LinearRegression(),
  'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
  'Gradient Boosting': GradientBoostingRegressor(n_estimators=100,
random_state=42)
}

# Cross-validation comparison
print('=== 5-FOLD CROSS-VALIDATION ===\n')
for name, model in models.items():
  pipe = Pipeline([
      ('preprocessor', preprocessor),
      ('model', model)
  ])
  cv_results = cross_validate(pipe, X, y, cv=5,
                               scoring=['r2', 'neg_mean_absolute_error'],
                               return_train_score=True)
  print(f'{name}:')
  print(f'  R² (test):  {cv_results["test_r2"].mean():.3f} ± {cv_results["test_r2"].std():.3f}')
  print(f'  MAE (test): ${abs(cv_results["test_neg_mean_absolute_error"].mean()):.2f}')
  print()

# Train best model on full data for feature importance
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
random_state=42)

best_pipe = Pipeline([
  ('preprocessor', preprocessor),
  ('model', GradientBoostingRegressor(n_estimators=100, random_state=42))
])
best_pipe.fit(X_train, y_train)
y_pred = best_pipe.predict(X_test)

print(f'=== BEST MODEL (Gradient Boosting) on TEST SET ===')
print(f'R²:  {r2_score(y_test, y_pred):.3f}')
print(f'MAE: ${mean_absolute_error(y_test, y_pred):.2f}')

# Feature importance
feature_names = numeric_features + list(best_pipe.named_steps['preprocessor']
                                       .named_transformers_['cat']

.get_feature_names_out(categorical_features))
importances = best_pipe.named_steps['model'].feature_importances_
feat_imp = pd.DataFrame({'feature': feature_names, 'importance':
importances}).sort_values('importance', ascending=False)
print('\n=== FEATURE IMPORTANCE ===')
print(feat_imp.to_string(index=False))

First 5 records:   ID\tYear_Birth\tEducation\tMarital_Status\tIncome\tKidhome\tTeenhome\tDt_Customer\tRecency\tMntWines\tMntFruits\tMntMeatProducts\tMntFishProducts\tMntSweetProducts\tMntGoldProds\tNumDealsPurchases\tNumWebPurchases\tNumCatalogPurchases\tNumStorePurchases\tNumWebVisitsMonth\tAcceptedCmp3\tAcceptedCmp4\tAcceptedCmp5\tAcceptedCmp1\tAcceptedCmp2\tComplain\tZ_CostContact\tZ_Revenue\tResponse
0  5524\t1957\tGraduation\tSingle\t58138\t0\t0\t0...                                                                                                                                                                                                                                                                                                                                                  
1  2174\t1954\tGraduation\tSingle\t46344\t1\t1\t0...                                                                                                                                                     

KeyError: ['Income']

In [2]:
# Predict CLV for all customers
df['Predicted_CLV'] = best_pipe.predict(X)

# Add clusters (same as your local notebook)
from sklearn.cluster import KMeans
cluster_features = ['Income', 'Recency', 'NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth', 'Age']
X_cluster = StandardScaler().fit_transform(df[cluster_features])
df['Cluster'] = KMeans(n_clusters=4, random_state=42,
n_init=10).fit_predict(X_cluster)

print('=== PREDICTED CLV BY CLUSTER ===')
print(df.groupby('Cluster').agg(
  count=('Predicted_CLV', 'count'),
  avg_predicted_clv=('Predicted_CLV', 'mean'),
  avg_actual_spend=('Total_Spending', 'mean'),
  avg_income=('Income', 'mean'),
  response_rate=('Response', 'mean')
).round(2))

=== PREDICTED CLV BY CLUSTER ===
         count  avg_predicted_clv  avg_actual_spend  avg_income  response_rate
Cluster                                                                       
0          537             106.62            104.25    33703.45           0.14
1          498             762.09            775.32    56145.74           0.18
2          676            1235.12           1238.96    75995.76           0.23
3          505             131.53            130.00    36331.57           0.03
